In [5]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score

In [ ]:
%pip install catboost

### Trying Random Forest

In [6]:
data=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/Classifier_Data/bot_iot_dataset_preprocessed.csv")
data.head()

,stime,flgs_e,flgs_e s,flgs_e dS,flgs_e g,flgs_e *,flgs_eU,flgs_e &,flgs_e d,flgs_e F,flgs_e r,proto_tcp,proto_udp,proto_icmp,proto_arp,proto_ipv6-icmp,proto_rarp,proto_igmp,saddr_192.168.100.149,saddr_192.168.100.148,saddr_192.168.100.150,saddr_192.168.100.147,saddr_192.168.100.3,saddr_192.168.100.7,saddr_192.168.100.6,saddr_192.168.100.5,saddr_192.168.100.4,saddr_192.168.100.1,saddr_private,saddr_external,daddr_192.168.100.149,daddr_192.168.100.148,daddr_192.168.100.150,daddr_192.168.100.147,daddr_192.168.100.3,daddr_192.168.100.7,daddr_192.168.100.6,daddr_192.168.100.5,daddr_192.168.100.4,daddr_192.168.100.1,...,bytes,state_RST,state_CON,state_INT,state_FIN,state_REQ,state_URP,state_ECO,state_NRS,state_ACC,state_MAS,ltime,dur,mean,stddev,sum,min,max,spkts,dpkts,sbytes,dbytes,rate,srate,drate,tnbpsrcip,tnbpdstip,tnp_psrcip,tnp_pdstip,tnp_perproto,tnp_per_dport,ar_p_proto_p_srcip,ar_p_proto_p_dstip,n_in_conn_p_srcip,n_in_conn_p_dstip,ar_p_proto_p_sport,ar_p_proto_p_dport,pkts_p_state_p_protocol_p_destip,pkts_p_state_p_protocol_p_srcip,attack_type
0,1.526963e+09,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,...,120,1,0,0,0,0,0,0,0,0,0,1.526963e+09,0.004128,0.004128,0.00000,0.004128,0.004128,0.004128,1,1,60,60,242.248077,0.000000,0.000000,826213862,9383605539,69025,9746137,11220924,8,4918.426175,939.965691,641.0,5208,564.567137,10430.133552,6482,23345,3
1,1.528081e+09,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,...,616,0,0,0,0,1,0,0,0,0,0,1.528081e+09,31.771238,0.000000,0.00000,0.000000,0.000000,0.000000,4,0,616,0,0.094425,0.094425,0.000000,5330629,5367610,42572,32388,11220924,13108094,3908.819243,1254.250649,120.0,6669,0.094425,100.862697,12994,15429,1
2,1.526982e+09,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,...,120,1,0,0,0,0,0,0,0,0,0,1.526982e+09,0.000224,0.000224,0.00000,0.000224,0.000224,0.000224,1,1,60,60,4464.285645,0.000000,0.000000,5330629,5367610,42572,32388,11220924,21,3908.819243,1254.250649,120.0,6669,4368.510051,18097.726590,13229,23301,3
3,1.528081e+09,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,...,890,1,0,0,0,0,0,0,0,0,0,1.528081e+09,31.326099,0.017032,0.02086,0.085161,0.000000,0.042693,5,2,770,120,0.191534,0.127863,0.048721,5669920,823284939,43720,52257,11220924,13108094,4574.700595,9901.705447,146.0,5063,0.191534,100.862697,14111,30428,1
4,1.526982e+09,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,...,120,1,0,0,0,0,0,0,0,0,0,1.526982e+09,0.000566,0.000566,0.00000,0.000566,0.000566,0.000566,1,1,60,60,1766.784424,0.000000,0.000000,826213862,5367610,69025,32388,11220924,23,4918.426175,1254.250649,641.0,6669,1856.201331,5937.703676,13229,23345,3


In [7]:
#splitting data into X and y
X=data.drop(["attack_type"],axis=1)
y=data["attack_type"]

In [8]:
y.head()

0    3
1    1
2    3
3    1
4    3
Name: attack_type, dtype: int64

In [9]:
#Splitting into train, val and test
np.random.seed(62)
X_train,X_test_val,y_train,y_test_val=train_test_split(X,y,test_size=0.4)
X_val,X_test, y_val,y_test=train_test_split(X_test_val,y_test_val, test_size=0.5)

In [10]:
len(y_train)+len(y_test)+len(y_val),len(data)

(49318, 49318)

In [8]:
#baseline random forest model
model=RandomForestClassifier()
model.fit(X_train,y_train)

RandomForestClassifier()

In [10]:
model.score(X_test,y_test)

0.9970600162206001

In [11]:
confusion_matrix(y_test,model.predict(X_test))

array([[1969,    0,    0,    0,    0,    0,    0],
       [   0, 1860,    0,    0,    0,    0,    0],
       [   0,    0, 1906,    0,    0,    0,    0],
       [   1,    0,    0, 1850,    7,    0,    0],
       [   0,    0,    0,   18, 1916,    0,    0],
       [   0,    0,    0,    1,    0,  304,    0],
       [   1,    0,    0,    0,    0,    1,   30]])

In [12]:
scores = cross_val_score(model, X_train, y_train, cv=5)
print(f"Cross-Validation Scores: {scores}")

Cross-Validation Scores: [0.99628253 0.99560662 0.99628253 0.99712741 0.99678946]


#### Feature Selection 

In [13]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
X_train_rf=pd.DataFrame()
best_score=0.0
for feature in X_train.columns :
    columns=list(X_train_rf.columns)
    columns.append(feature)
    X_train_rf=X_train[columns]
    model=RandomForestClassifier()
    mean_score = np.mean(cross_val_score(model, X_train_rf, y_train, cv=5))
    if mean_score>=best_score :
       best_score=mean_score
    else :
        X_train_rf.drop(feature,axis=1,inplace=True)

print(best_score)

0.9991213247718823


In [15]:
X_train_rf.columns

Index(['stime', 'flgs_e', 'flgs_e dS', 'flgs_e g', 'flgs_e r', 'proto_tcp',
       'proto_udp', 'proto_icmp', 'proto_arp', 'proto_ipv6-icmp',
       'saddr_192.168.100.150', 'saddr_192.168.100.3', 'saddr_192.168.100.6',
       'saddr_192.168.100.5', 'saddr_192.168.100.4', 'saddr_192.168.100.1',
       'saddr_private', 'daddr_192.168.100.149', 'daddr_192.168.100.148',
       'daddr_192.168.100.150', 'daddr_192.168.100.147', 'daddr_192.168.100.3',
       'daddr_192.168.100.7', 'daddr_192.168.100.6', 'daddr_192.168.100.5',
       'daddr_192.168.100.4', 'daddr_192.168.100.1', 'daddr_private',
       'daddr_external', 'pkts', 'state_URP', 'state_ECO', 'state_ACC',
       'ltime', 'sum', 'tnbpdstip', 'ar_p_proto_p_sport'],
      dtype='object')

In [16]:
selected_columns=['stime', 'flgs_e', 'flgs_e dS', 'flgs_e g', 'flgs_e r', 'proto_tcp',
       'proto_udp', 'proto_icmp', 'proto_arp', 'proto_ipv6-icmp',
       'saddr_192.168.100.150', 'saddr_192.168.100.3', 'saddr_192.168.100.6',
       'saddr_192.168.100.5', 'saddr_192.168.100.4', 'saddr_192.168.100.1',
       'saddr_private', 'daddr_192.168.100.149', 'daddr_192.168.100.148',
       'daddr_192.168.100.150', 'daddr_192.168.100.147', 'daddr_192.168.100.3',
       'daddr_192.168.100.7', 'daddr_192.168.100.6', 'daddr_192.168.100.5',
       'daddr_192.168.100.4', 'daddr_192.168.100.1', 'daddr_private',
       'daddr_external', 'pkts', 'state_URP', 'state_ECO', 'state_ACC',
       'ltime', 'sum', 'tnbpdstip', 'ar_p_proto_p_sport']

In [17]:
model=RandomForestClassifier()
model.fit(X_train[selected_columns],y_train)
model.score(X_test[selected_columns],y_test)

0.9988848337388483

In [18]:
y_pred_proba=model.predict_proba(X_test[selected_columns])

In [19]:
y_pred_proba[:5]

array([[0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.]])

In [20]:
print(classification_report(y_test, model.predict(X_test[selected_columns])))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1969
           1       1.00      1.00      1.00      1860
           2       1.00      1.00      1.00      1906
           3       1.00      1.00      1.00      1858
           4       1.00      1.00      1.00      1934
           5       1.00      1.00      1.00       305
           6       1.00      0.94      0.97        32

    accuracy                           1.00      9864
   macro avg       1.00      0.99      0.99      9864
weighted avg       1.00      1.00      1.00      9864



In [22]:
confusion_matrix(y_test,model.predict(X_test[selected_columns]))

array([[1969,    0,    0,    0,    0,    0,    0],
       [   0, 1860,    0,    0,    0,    0,    0],
       [   0,    0, 1906,    0,    0,    0,    0],
       [   1,    0,    0, 1853,    4,    0,    0],
       [   0,    0,    0,    4, 1930,    0,    0],
       [   0,    0,    0,    0,    0,  305,    0],
       [   1,    0,    0,    0,    0,    1,   30]])

In [23]:
y_test.head()

28341    1
22567    1
45645    1
4001     4
26601    1
Name: attack_type, dtype: int64

### XGBoost

In [24]:
model=XGBClassifier()
model.fit(X_train,y_train,verbose=True)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [26]:
model.score(X_val,y_val)

0.9987834549878345

In [11]:
X_train_xg=pd.DataFrame()
best_score=0.0
for feature in X_train.drop(["stime","ltime","stddev"],axis=1).columns :
    columns=list(X_train_xg.columns)
    columns.append(feature)
    X_train_xg=X_train[columns]
    model=XGBClassifier()
    mean_score = np.mean(cross_val_score(model, X_train_xg, y_train, cv=5))
    if mean_score>=best_score :
       best_score=mean_score
    else :
        X_train_xg.drop(feature,axis=1,inplace=True)

print(best_score)

/tmp/ipykernel_4849/82184564.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_xg.drop(feature,axis=1,inplace=True)
/tmp/ipykernel_4849/82184564.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_xg.drop(feature,axis=1,inplace=True)
/tmp/ipykernel_4849/82184564.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_xg.drop(feature,axis=1,inplace=True)
/tmp/ipykernel_4849/82184564.py:12: SettingWithCopyWarning

0.980770530584657


/tmp/ipykernel_4849/82184564.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_xg.drop(feature,axis=1,inplace=True)


In [12]:
X_train_xg.columns

Index(['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
       'flgs_e &', 'flgs_e    F', 'flgs_e r', 'proto_tcp', 'proto_udp',
       'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
       'proto_igmp', 'saddr_192.168.100.149', 'saddr_192.168.100.148',
       'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
       'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'daddr_192.168.100.149', 'daddr_192.168.100.150',
       'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
       'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4',
       'daddr_private', 'daddr_external', 'pkts', 'bytes', 'state_RST',
       'state_CON', 'state_NRS', 'state_ACC', 'state_MAS', 'dur', 'mean',
       'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
       'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip'],
      dtype='object')

In [13]:
selected_columns=X_train_xg.columns

In [14]:
model=XGBClassifier()
model.fit(X_train_xg,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [15]:
model.score(X_test[selected_columns],y_test)

0.9788118410381184

#### Hyperparameter Tuning

In [16]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

In [18]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [0, 0.01, 0.1, 1.0],
    'scale_pos_weight': [1, 3, 5],
    'booster': ['gbtree', 'gblinear', 'dart']
}
model=RandomizedSearchCV(estimator=XGBClassifier(),param_distributions=xgb_param_grid,cv=5,verbose=2,n_iter=1000)
model.fit(X_train_xg,y_train)